# Fixed Threshold Electricity Trading

This notebook simulates a simple battery arbitrage rule: buy electricity when the price is at or below a fixed threshold, sell electricity when the price is at or above a fixed threshold, and hold otherwise. It can load either the historical price file or the generated stochastic price model, applies battery power and state-of-charge limits, and plots the resulting dispatch.

In [1]:
import os
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "Battery_model").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the THESIS repository root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.battery_model.price_model import DEFAULT_PRICE_MODEL_HOURS, load_price_model_data


PRICE_FILE = ROOT / "Battery_model" / "Optimization_Pyomo" / "input_data" / "electricity_price.txt"
OUTPUT_FIGURE = ROOT / "notebooks" / "fixed_threshold_dispatch.png"

PRICE_SOURCE = "price_model"  # "file" or "price_model"
PRICE_MODEL_HOURS = DEFAULT_PRICE_MODEL_HOURS
PRICE_MODEL_SEED = 42
PRICE_MODEL_INITIAL_PRICE = None
PRICE_MODEL_START_HOUR = 0.0
PRICE_MODEL_DT_HOURS = 1.0

WINDOW_START = 23
WINDOW_HOURS = 24

BUY_THRESHOLD_EUR_PER_MWH = -25
SELL_THRESHOLD_EUR_PER_MWH = 0.0
P_MAX_KW = 20.0
ETA = 0.90
E_MAX_KWH = 200.0
SOC_0 = 0.50

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ROOT

PosixPath('/home/ashish/Documents/THESIS')

In [2]:
if PRICE_SOURCE == "file":
    price_df = pd.read_csv(
        PRICE_FILE,
        sep=r"\s+",
        header=None,
        names=["timestamp_s", "price_eur_per_mwh"],
    )
    price_df["hour"] = np.arange(len(price_df))
    prices_full = price_df["price_eur_per_mwh"].astype(float).to_numpy()
    price_source_label = str(PRICE_FILE.relative_to(ROOT))
elif PRICE_SOURCE == "price_model":
    prices_full, _ = load_price_model_data(
        num_hours=PRICE_MODEL_HOURS,
        initial_price=PRICE_MODEL_INITIAL_PRICE,
        start_hour=PRICE_MODEL_START_HOUR,
        dt_hours=PRICE_MODEL_DT_HOURS,
        seed=PRICE_MODEL_SEED,
    )
    price_df = pd.DataFrame({
        "hour": np.arange(len(prices_full)),
        "price_eur_per_mwh": prices_full,
    })
    price_source_label = f"price_model(seed={PRICE_MODEL_SEED})"
else:
    raise ValueError('PRICE_SOURCE must be "file" or "price_model".')

window_end = min(WINDOW_START + WINDOW_HOURS, len(prices_full))
prices_window = prices_full[WINDOW_START:window_end]
window_df = price_df.iloc[WINDOW_START:window_end].copy()

print(f"Price source: {PRICE_SOURCE}")
print(f"Loaded {len(prices_full)} hourly prices from {price_source_label}")
print(f"Using hours {WINDOW_START} to {window_end - 1} ({len(prices_window)} hours)")
print(f"Buy threshold: {BUY_THRESHOLD_EUR_PER_MWH:.2f} EUR/MWh")
print(f"Sell threshold: {SELL_THRESHOLD_EUR_PER_MWH:.2f} EUR/MWh")

window_df.head()

Generated price-model dataset length: 8784 hours
Global price scale (95th pct): 54.8514
Price-model residual shock std (1 h): 4.7474 EUR/MWh
Price source: price_model
Loaded 8784 hourly prices from price_model(seed=42)
Using hours 23 to 46 (24 hours)
Buy threshold: -25.00 EUR/MWh
Sell threshold: 0.00 EUR/MWh


,hour,price_eur_per_mwh
23,23,12.537361
24,24,28.531410
25,25,41.354064
26,26,49.108128
27,27,53.215399


In [3]:
def build_dispatch_frame(prices, power_kw, power_actual_kw, soc, actions):
    return pd.DataFrame({
        "hour": np.arange(len(prices)),
        "price_eur_per_mwh": np.asarray(prices, dtype=float),
        "power_kw": np.asarray(power_kw, dtype=float),
        "power_actual_kw": np.asarray(power_actual_kw, dtype=float),
        "charge_kw": np.clip(np.asarray(power_kw, dtype=float), 0.0, None),
        "discharge_kw": np.clip(-np.asarray(power_kw, dtype=float), 0.0, None),
        "soc": np.asarray(soc, dtype=float),
        "action": np.asarray(actions, dtype=float),
    })


def simulate_fixed_threshold_dispatch(
    prices,
    buy_threshold,
    sell_threshold,
    p_max_kw=P_MAX_KW,
    eta=ETA,
    e_max_kwh=E_MAX_KWH,
    soc_0=SOC_0,
):
    if buy_threshold >= sell_threshold:
        raise ValueError("buy_threshold must be smaller than sell_threshold.")

    soc = float(soc_0)
    dt_hours = 1.0
    power_kw = []
    power_actual_kw = []
    soc_values = []
    actions = []

    for price in prices:
        if price <= buy_threshold:
            action = 1.0
        elif price >= sell_threshold:
            action = -1.0
        else:
            action = 0.0

        requested_power = action * p_max_kw
        room_for_charge = (1.0 - soc) * e_max_kwh
        room_for_discharge = soc * e_max_kwh
        max_charge_kw = min(p_max_kw, (room_for_charge / dt_hours) / max(eta, 1e-6))
        max_discharge_kw = min(p_max_kw, (room_for_discharge / dt_hours) * max(eta, 1e-6))
        constrained_power = float(np.clip(requested_power, -max_discharge_kw, max_charge_kw))

        efficiency_factor = 0.5 * ((1.0 + np.tanh(constrained_power)) * eta + (1.0 - np.tanh(constrained_power)) / eta)
        actual_power = float(constrained_power * efficiency_factor)
        soc = float(np.clip(soc + actual_power * dt_hours / e_max_kwh, 0.0, 1.0))

        power_kw.append(constrained_power)
        power_actual_kw.append(actual_power)
        soc_values.append(soc)
        actions.append(action)

    return build_dispatch_frame(prices, power_kw, power_actual_kw, soc_values, actions)


def objective_value_eur(df):
    return float(np.sum((df["price_eur_per_mwh"] / 1000.0) * df["power_kw"]))


def summarize_dispatch(df):
    return pd.DataFrame([
        {
            "buy_threshold_eur_per_mwh": BUY_THRESHOLD_EUR_PER_MWH,
            "sell_threshold_eur_per_mwh": SELL_THRESHOLD_EUR_PER_MWH,
            "window_hours": len(df),
            "buy_energy_kwh": float(df["charge_kw"].sum()),
            "sell_energy_kwh": float(df["discharge_kw"].sum()),
            "objective_value_eur": objective_value_eur(df),
            "min_soc": float(df["soc"].min()),
            "max_soc": float(df["soc"].max()),
        }
    ])

In [4]:
dispatch_df = simulate_fixed_threshold_dispatch(
    prices_window,
    BUY_THRESHOLD_EUR_PER_MWH,
    SELL_THRESHOLD_EUR_PER_MWH,
)

summary_df = summarize_dispatch(dispatch_df)
print(
    "Fixed threshold rule: buy when price <= {0:.2f} EUR/MWh, sell when price >= {1:.2f} EUR/MWh".format(
        BUY_THRESHOLD_EUR_PER_MWH, SELL_THRESHOLD_EUR_PER_MWH
    )
)
display(summary_df)
dispatch_df.head()

Fixed threshold rule: buy when price <= -25.00 EUR/MWh, sell when price >= 0.00 EUR/MWh


,buy_threshold_eur_per_mwh,sell_threshold_eur_per_mwh,window_hours,buy_energy_kwh,sell_energy_kwh,objective_value_eur,min_soc,max_soc
0,-25,0.0,24,0.0,90.0,-3.162773,1.116859e-24,0.388889


,hour,price_eur_per_mwh,power_kw,power_actual_kw,charge_kw,discharge_kw,soc,action
0,0,12.537361,-20.0,-22.222222,0.0,20.0,3.888889e-01,-1.0
1,1,28.531410,-20.0,-22.222222,0.0,20.0,2.777778e-01,-1.0
2,2,41.354064,-20.0,-22.222222,0.0,20.0,1.666667e-01,-1.0
3,3,49.108128,-20.0,-22.222222,0.0,20.0,5.555556e-02,-1.0
4,4,53.215399,-10.0,-11.111111,0.0,10.0,2.175662e-11,-1.0


In [5]:
def plot_fixed_threshold_dispatch(df, output_path=None):
    hours = df["hour"].to_numpy()
    x_min = int(hours.min())
    x_max = int(hours.max())

    fig, (ax_power, ax_soc) = plt.subplots(
        2,
        1,
        figsize=(14, 5.8),
        sharex=True,
        gridspec_kw={"height_ratios": [3.0, 1.0], "hspace": 0.08},
    )

    charge_power = df["charge_kw"]
    discharge_power = -df["discharge_kw"]

    ax_power.fill_between(hours, 0, charge_power, step="post", color="#0b5d1e", alpha=0.45, label="Charge power")
    ax_power.fill_between(hours, 0, discharge_power, step="post", color="#0b5d1e", alpha=0.50, label="Discharge power")
    ax_power.step(hours, charge_power, where="post", color="#0b5d1e", linewidth=1.1, label="_nolegend_")
    ax_power.step(hours, discharge_power, where="post", color="#0b5d1e", linewidth=1.1, label="_nolegend_")
    ax_power.axhline(0.0, color="#444444", linewidth=0.8)
    ax_power.set_ylabel("Power [kW]", fontweight="bold")
    ax_power.set_xlim(x_min, x_max)
    ax_power.grid(True, alpha=0.28)

    ax_price = ax_power.twinx()
    ax_price.plot(hours, df["price_eur_per_mwh"], color="#b71c1c", linewidth=2.0, label="Electricity price")
    ax_price.axhline(BUY_THRESHOLD_EUR_PER_MWH, color="#1b5e20", linestyle="--", linewidth=1.6, label="Buy threshold")
    ax_price.axhline(SELL_THRESHOLD_EUR_PER_MWH, color="#7f0000", linestyle="--", linewidth=1.6, label="Sell threshold")
    ax_price.set_ylabel("Electricity price [EUR/MWh]", fontweight="bold")

    ax_soc.plot(hours, df["soc"] * 100.0, color="#1f1f1f", linewidth=1.9, label="State of charge")
    ax_soc.fill_between(hours, 0, df["soc"] * 100.0, color="#1f1f1f", alpha=0.08)
    ax_soc.set_ylim(0, 100)
    ax_soc.set_ylabel("SOC [%]", fontweight="bold")
    ax_soc.set_xlabel("Time [h]")
    ax_soc.grid(True, alpha=0.28)

    power_handles, power_labels = ax_power.get_legend_handles_labels()
    price_handles, price_labels = ax_price.get_legend_handles_labels()
    soc_handles, soc_labels = ax_soc.get_legend_handles_labels()

    title = (
        f"Fixed threshold battery dispatch | hours {WINDOW_START}-{WINDOW_START + len(df) - 1}"
    )
    objective_text = (
        f"Objective value: {objective_value_eur(df):.2f} EUR"
    )

    fig.suptitle(title, y=0.98)
    fig.text(0.5, 0.925, objective_text, ha="center", va="center", fontsize=9)
    fig.legend(
        power_handles + price_handles + soc_handles,
        power_labels + price_labels + soc_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.895),
        frameon=True,
        ncol=5,
        fontsize=9,
    )
    fig.subplots_adjust(top=0.80)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to {output_path.relative_to(ROOT)}")

    return fig


plot_fixed_threshold_dispatch(dispatch_df, OUTPUT_FIGURE)

Saved figure to notebooks/fixed_threshold_dispatch.png


<Figure size 1680x696 with 3 Axes>

To change the behavior, adjust `BUY_THRESHOLD_EUR_PER_MWH` and `SELL_THRESHOLD_EUR_PER_MWH` in the second code cell. Lower buy thresholds make charging more selective, while higher sell thresholds make discharging more selective. You can also change `WINDOW_START` and `WINDOW_HOURS` to inspect a different period. For generated prices, edit `RESIDUAL_VOLATILITY_SCALE` or `PRICE_LEVEL_SHIFT` in `src/battery_model/price_model.py`.